# 01 - Cleaning & Integration (A1-A8)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
SEED = 42

## A0. Load the raw files
We inspect first; we never edit `data/raw/` by hand.

In [2]:
train_raw=pd.read_csv(ROOT/'data/raw/crop_yield_train.csv')
test_raw=pd.read_csv(ROOT/'data/raw/crop_yield_leaderboard_test.csv')
weather_raw=pd.read_csv(ROOT/'data/raw/regional_weather.csv')
price_raw=pd.read_csv(ROOT/'data/raw/market_prices.csv')
print(train_raw.shape,test_raw.shape,weather_raw.shape,price_raw.shape)
train_raw.head()

(15090, 15) (3750, 14) (232, 6) (100, 4)


,plot_id,region,crop_type,survey_year,planting_month,altitude_m,rainfall_mm_season,farm_size_ha,fertilizer_kg_per_ha,improved_seed_used,pest_disease_flag,soil_quality_index,labor_days_per_ha,distance_to_market_km,yield_tons_per_ha
0,PLOT-000339,OROMIA,barley,2024,Aug,2697.817349,1157.403560,0.867604,29.393684,0,1,0.399408,42.678102,8.573726,2.209711
1,PLOT-007457,Amhara,Maize,2023,Jun,1523.350141,897.420122,1.099197,16.843992,1,0,0.314488,45.370495,19.928302,3.363254
2,PLOT-006092,OROMIA,wheat,2024,Feb,2389.441948,1119.155741,0.811309,41.015144,0,0,0.812596,52.846126,8.961797,3.560614
3,PLOT-004128,SNNPR,barley,2022,Aug,2612.483420,1246.462210,1.602312,14.353759,0,1,0.619516,59.833780,1.849243,1.924800
4,PLOT-003033,Amhara,sorghum,2021,Jul,1210.095040,928.477088,0.937758,33.736138,1,0,0.577963,22.020843,3.149654,4.171220


## A1. Find the problems
Missing values, `-999`, category spelling, duplicates and suspicious price units are intentional hackathon issues.

In [3]:
for name,d in {'train':train_raw,'test':test_raw,'weather':weather_raw,'price':price_raw}.items():
    print('\n',name,'missing=',int(d.isna().sum().sum()),'duplicates=',d.duplicated().sum())
    nums=d.select_dtypes('number')
    print('-999=',int((nums==-999).sum().sum()))


 train missing= 1805 duplicates= 0
-999= 788

 test missing= 439 duplicates= 0
-999= 180

 weather missing= 3 duplicates= 1
-999= 0

 price missing= 4 duplicates= 0
-999= 0


### A1.1 Automated suspicious-range / outlier screen
This diagnostic does **not** delete rows automatically. It flags unusual numeric values so cleaning decisions are transparent and reproducible. `-999` sentinels are reported separately from statistical IQR flags.


In [4]:
# A1.1 - transparent suspicious-value and statistical outlier screen
plot_numeric = [
    'altitude_m','rainfall_mm_season','farm_size_ha','fertilizer_kg_per_ha',
    'soil_quality_index','labor_days_per_ha','distance_to_market_km'
]

def outlier_screen(frame, dataset_name):
    rows = []
    for col in [c for c in plot_numeric if c in frame.columns]:
        s = pd.to_numeric(frame[col], errors='coerce')
        valid = s[(s != -999) & s.notna()]
        if valid.empty:
            continue
        q1, q3 = valid.quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        rows.append({
            'dataset': dataset_name, 'column': col,
            'missing': int(s.isna().sum()), 'sentinel_-999': int((s == -999).sum()),
            'iqr_low': round(float(lo), 3), 'iqr_high': round(float(hi), 3),
            'iqr_flags': int(((valid < lo) | (valid > hi)).sum())
        })
    return pd.DataFrame(rows)

outlier_audit = pd.concat([
    outlier_screen(train_raw, 'train'),
    outlier_screen(test_raw, 'test')
], ignore_index=True)

display(outlier_audit)
print('Note: IQR flags are review signals, not automatic errors. Domain-valid extremes are retained unless a documented cleaning rule says otherwise.')


,dataset,column,missing,sentinel_-999,iqr_low,iqr_high,iqr_flags
0,train,altitude_m,0,0,-290.945,4281.190,0
1,train,rainfall_mm_season,449,0,252.518,1449.849,149
2,train,farm_size_ha,0,0,-0.506,2.859,692
3,train,fertilizer_kg_per_ha,748,0,-30.571,103.503,459
4,train,soil_quality_index,608,0,-0.084,1.202,0
5,train,labor_days_per_ha,0,488,4.254,85.650,57
6,train,distance_to_market_km,0,0,-16.058,35.923,732
7,test,altitude_m,0,0,-239.293,4243.990,0
8,test,rainfall_mm_season,125,0,256.497,1444.696,31
9,test,farm_size_ha,0,0,-0.496,2.810,174


Note: IQR flags are review signals, not automatic errors. Domain-valid extremes are retained unless a documented cleaning rule says otherwise.


## A2. Standardize keys
This is deterministic cleaning, not learned preprocessing.

In [5]:
def clean_region(s):
    x=s.astype('string').str.strip().str.lower()
    return x.replace({'oro':'oromia','amh':'amhara','tig':'tigray','som':'somali','snnp':'snnpr'}).str.title().replace({'Snnpr':'SNNPR'})
def clean_crop(s):
    return s.astype('string').str.strip().str.lower().replace({'tef':'teff'})
print('Before regions:',train_raw.region.unique()[:12])
print('After regions:',clean_region(train_raw.region).unique())
print('Before crops:',train_raw.crop_type.unique()[:12])
print('After crops:',clean_crop(train_raw.crop_type).unique())

Before regions: ['OROMIA' 'Amhara' 'SNNPR' 'SOMALI' 'Tigray' 'SNNPR ' 'Somali' 'Oromia'
 'oromia' 'amhara' 'tigray' 'AMHARA']
After regions: <StringArray>
['Oromia', 'Amhara', 'SNNPR', 'Somali', 'Tigray']
Length: 5, dtype: string
Before crops: ['barley' 'Maize' 'wheat' 'sorghum' 'Sorghum' 'teff' 'maize' ' teff '
 'Tef' ' maize ' ' wheat ' 'MAIZE']
After crops: <StringArray>
['barley', 'maize', 'wheat', 'sorghum', 'teff']
Length: 5, dtype: string


## A3-A5. Weather join rule
We use **planting month + following 3 months**. The plot table is the left table, so no farm disappears. The full executed audit and 3-plot proof are saved in `reports/`.

In [6]:
pd.read_csv(ROOT/'reports/join_audit.csv')

,join,rows_before,rows_after,match_rate_pct,unmatched,incomplete_weather_windows
0,train->weather,15090,15090,100.0,0,2925
1,test->weather,3750,3750,100.0,0,748
2,train->price,15090,15090,100.0,0,0
3,test->price,3750,3750,100.0,0,0


In [7]:
pd.read_csv(ROOT/'reports/join_proof_3_plots.csv').head(12)

,_row,region,survey_year,_pm,weather_year,month_num,month,offset,year,avg_temp_c,monthly_rainfall_mm,extreme_heat_days,plot_id
0,0,Oromia,2024,8,2024,8,Aug,0,2024.0,19.1,185.1,0.0,PLOT-000339
1,1,Amhara,2023,6,2023,6,Jun,0,2023.0,17.0,118.5,0.0,PLOT-007457
2,2,Oromia,2024,2,2024,2,Feb,0,2024.0,18.2,78.5,0.0,PLOT-006092
3,0,Oromia,2024,8,2024,9,Sep,1,2024.0,15.1,147.1,0.0,PLOT-000339
4,1,Amhara,2023,6,2023,7,Jul,1,2023.0,14.9,244.6,0.0,PLOT-007457
5,2,Oromia,2024,2,2024,3,Mar,1,2024.0,20.1,92.5,0.0,PLOT-006092
6,0,Oromia,2024,8,2024,10,Oct,2,NaN,NaN,NaN,NaN,PLOT-000339
7,1,Amhara,2023,6,2023,8,Aug,2,2023.0,14.5,192.8,0.0,PLOT-007457
8,2,Oromia,2024,2,2024,4,Apr,2,2024.0,20.2,55.8,0.0,PLOT-006092
9,0,Oromia,2024,8,2024,11,Nov,3,2024.0,19.4,20.8,0.0,PLOT-000339


## A6. Engineered features
At least 3 weather features + an interaction + planting-month feature are required.

In [8]:
pd.read_csv(ROOT/'reports/feature_engineering_table.csv')

,feature,formula,source,why
0,season_avg_temp_c,mean(avg_temp_c over 4 months),weather,captures thermal growing conditions
1,season_rainfall_mm,sum(monthly_rainfall_mm over 4 months),weather,captures seasonal water supply
2,season_extreme_heat_days,sum(extreme_heat_days over 4 months),weather,captures heat stress
3,season_temp_deviation_c,season temp - region/year typical,weather,captures anomaly
4,planting_month_num,Jan=1 ... Dec=12,planting_month,numeric timing signal
5,fertilizer_seed_interaction,fertilizer × improved_seed,plot,management interaction
6,rainfall_gap_mm,reported rainfall - station rainfall,plot + weather,measurement/local climate difference


## A7. Automated integrity checks

In [9]:
pd.read_csv(ROOT/'reports/integrity_checks.csv')

,check,result,status
0,train plot_id unique,True,PASS
1,test plot_id unique,True,PASS
2,train row count unchanged,True,PASS
3,test row count unchanged,True,PASS
4,allowed regions,True,PASS
5,allowed crops,True,PASS
6,yield nonnegative,True,PASS
7,price positive,True,PASS
8,train/test feature columns align,True,PASS


## A8. Master tables
These are the outputs used by analysis and modeling. Missing numeric values are intentionally left for the **training-only model imputer** rather than filled using test statistics.

In [10]:
master_train=pd.read_csv(ROOT/'data/processed/master_train.csv')
master_test=pd.read_csv(ROOT/'data/processed/master_test.csv')
print(master_train.shape,master_test.shape)
master_train.head()

(15090, 24) (3750, 23)


,plot_id,region,crop_type,survey_year,planting_month,altitude_m,rainfall_mm_season,farm_size_ha,fertilizer_kg_per_ha,improved_seed_used,...,yield_tons_per_ha,season_avg_temp_c,season_rainfall_mm,season_extreme_heat_days,weather_months_found,season_temp_deviation_c,planting_month_num,fertilizer_seed_interaction,rainfall_gap_mm,price_birr_per_quintal
0,PLOT-000339,Oromia,barley,2024,Aug,2697.817349,1157.403560,0.867604,29.393684,0,...,2.209711,17.866667,353.0,0.0,3,-1.178986,8,0.000000,804.403560,4590.0
1,PLOT-007457,Amhara,maize,2023,Jun,1523.350141,897.420122,1.099197,16.843992,1,...,3.363254,15.200000,709.4,0.0,4,-1.882222,6,16.843992,188.020122,3073.5
2,PLOT-006092,Oromia,wheat,2024,Feb,2389.441948,1119.155741,0.811309,41.015144,0,...,3.560614,19.200000,294.9,0.0,4,0.154348,2,0.000000,824.255741,5387.0
3,PLOT-004128,SNNPR,barley,2022,Aug,2612.483420,1246.462210,1.602312,14.353759,0,...,1.924800,19.425000,398.3,1.0,4,-1.022826,8,0.000000,848.162210,4181.0
4,PLOT-003033,Amhara,sorghum,2021,Jul,1210.095040,928.477088,0.937758,33.736138,1,...,4.171220,16.500000,798.2,1.0,4,-0.582222,7,33.736138,130.277088,2706.0


### Leakage checkpoint
- We did **not** use test yield (it does not exist).
- We did **not** calculate model imputers/scalers/encoders from test.
- Price is present for analysis/demo but will be explicitly removed from model features.
- Weather features come from the external weather table.